# Federated AMR — Ceftazidime — Site-D Ablation (no-D arm)

**Leave-site-D-out federated run**: clients A/B/C only, site D kept in
evaluation as an unseen site. Configuration (splits, seed, hyperparameters,
thresholds, rounds) mirrors the with-D run `Drugs/Ceftazidime/03-Run`.

| Component | Detail |
|---|---|
| Drug | **Ceftazidime** |
| Train sites | A, B, C (site D excluded) |
| Eval sites | A, B, C (trained) + D (unseen) |
| Architecture | MLP: 6000 → 512 → 256 → 128 → 2 |
| Strategies | Centralized MLP/RF (pooled A/B/C), FedAvg MLP, FedAvg LR, FedRF |
| Checkpoints | Disabled (results only) |
| Mixing | Small sites train multiple seeds, average weights (MLP only) |

Flower simulation (Ray backend), local CPU. FedProx skipped (unstable at tuned μ).


In [ ]:
# (pip cell skipped for local execution)


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings, copy, os, io, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     RandomizedSearchCV, cross_val_predict,
                                     StratifiedKFold)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score, log_loss)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

import flwr as fl
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

FL_DEVICE = "cpu"  # Ray workers don't have GPU access
print(f"Flower version: {fl.__version__}")
print(f"FL device: {FL_DEVICE}")

In [ ]:
# ── Paths ──
DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftazidime"
DRUG_CSV  = "Ceftazidime"

OUT_DIR = Path("./results")
MODEL_DIR = Path("./models")   # checkpoints disabled in this arm; kept for parity
OUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
TRAIN_SITES = ["A", "B", "C"]     # ← Site-D ablation: D excluded from training
EVAL_SITES  = SITE_ORDER          # D stays in evaluation (unseen-site transfer)

ARCHIVE_RUN = "03-Run"             # with-D run this arm mirrors
ARCHIVE_PARAMS = (Path("../..") / "Drugs" / DRUG_NAME / ARCHIVE_RUN
                  / "results" / "best_params_used.txt")
BEST_PARAMS_CSV = DRYAD / "Processed/Processing/Analysis/07-Dedicated-MLP-Aggregated/07-01-results_dedicated_lr_mlp/best_params.csv"

print(f"Drug: {DRUG_NAME} | train sites: {TRAIN_SITES} | eval sites: {EVAL_SITES}")
print(f"Mirroring with-D run {ARCHIVE_RUN}: {ARCHIVE_PARAMS.resolve()}")


In [ ]:
# ── Shared constants ──
THRESHOLDS = np.linspace(0.05, 0.95, 91)
NUM_ROUNDS = 30
NUM_RF_ROUNDS = 5
LOCAL_EPOCHS = 1
BATCH_SIZE = 16
FEDPROX_MUS = [0.1]

RF_PARAM_GRID = {
    "n_estimators": [200, 300],
    "max_depth": [20, None],
    "min_samples_leaf": [5, 10],
    "class_weight": ["balanced", "balanced_subsample"],
}

def n_mixes(n_train):
    if n_train < 500: return 5
    if n_train < 1500: return 4
    if n_train < 5000: return 3
    return 1

def weighted_avg_loss(metrics_list):
    total = 0; weighted_sum = 0
    for item in metrics_list:
        n, d = item if isinstance(item, tuple) else (item.get("num_examples", 1), item)
        total += n; weighted_sum += d.get("train_loss", 0) * n
    return {"avg_train_loss": weighted_sum / total, "total_examples": total} if total > 0 else {}

In [ ]:
# ── Load pre-computed LR + MLP params from 07-01 ──
params_df = pd.read_csv(BEST_PARAMS_CSV)
row = params_df[params_df["Drug"] == DRUG_CSV]
if len(row) == 0:
    print(f"WARNING: drug '{DRUG_CSV}' not found in best_params.csv. Using defaults.")
    BEST_LR_C = 2.5e-04; BEST_MLP_LR = 8.2e-05; BEST_MLP_DH = 0.7
else:
    r = row.iloc[0]
    BEST_LR_C = float(r["LR"].replace("C=", ""))
    mlp = r["MLP"]
    BEST_MLP_LR = float(mlp.split("lr=")[1].split(" ")[0])
    BEST_MLP_DH = float(mlp.split("drop=")[1])
print(f"  LR:   C={BEST_LR_C:.2e}")
print(f"  MLP:  lr={BEST_MLP_LR:.2e}  dropout={BEST_MLP_DH:.1f}")

In [ ]:
# ── Load drug CSV from all 4 sites ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    n_r, n_s = (y == 1).sum(), (y == 0).sum()
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples ({n_s} S, {n_r} R, {n_r/len(y)*100:.1f}% R, {n_sp} species)")

total = sum(len(raw_data[s][1]) for s in SITE_ORDER)
print(f"\nTotal pooled: {total} samples")

In [ ]:
# ── Per-site species-stratified 90/10 train/test ──
client_train = {}
client_test = {}
species_train = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    n = len(y)
    idx = np.arange(n).reshape(-1, 1)
    n_strata = len(set(zip(sp.tolist(), y.tolist())))
    test_size = max(0.10, (n_strata + 1) / n)
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx, y, species=sp, test_size=test_size, random_state=SEED)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)
    client_train[site] = (X[idx_train], y[idx_train])
    client_test[site] = (X[idx_val], y[idx_val])
    species_train[site] = sp[idx_train]
    print(f"  Site {site}: train={len(idx_train)} test={len(idx_val)} (test_size={test_size:.2f})")

pooled_X_train = np.concatenate([client_train[s][0] for s in TRAIN_SITES])
pooled_y_train = np.concatenate([client_train[s][1] for s in TRAIN_SITES])
print(f"\nPooled train: {len(pooled_X_train)} (for cross-site baseline)")

In [ ]:
# ── Downsample training data (keep all R, cap S in bad-ratio species) ──
TRAIN_DS_S_PER_R = 10   # global: all training data (bad-ratio species only)

def downsample_keep_idx(y, sp, s_per_r, bad_ratio=0.10, min_n=400, rng=None):
    rng = rng if rng is not None else np.random.default_rng(SEED)
    idx = np.arange(len(y))
    keep = idx[y == 1].tolist()
    for spec in np.unique(sp):
        m = sp == spec
        n_r = int((m & (y == 1)).sum()); n_s = int((m & (y == 0)).sum())
        total = n_r + n_s
        s_pos = idx[m & (y == 0)]
        if n_r > 0 and total > min_n and (n_r / total) < bad_ratio and n_s > n_r * s_per_r:
            keep.extend(rng.choice(s_pos, size=int(round(n_r * s_per_r)), replace=False).tolist())
        else:
            keep.extend(s_pos.tolist())
    return np.sort(np.array(keep, dtype=int))

_rng_ds = np.random.default_rng(SEED)
for site in SITE_ORDER:
    X, y = client_train[site]; sp = species_train[site]
    keep = downsample_keep_idx(y, sp, TRAIN_DS_S_PER_R, rng=_rng_ds)
    client_train[site] = (X[keep], y[keep]); species_train[site] = sp[keep]
    print(f"  Site {site}: kept {len(keep)}/{len(y)} train")

pooled_X_train = np.concatenate([client_train[s][0] for s in TRAIN_SITES])
pooled_y_train = np.concatenate([client_train[s][1] for s in TRAIN_SITES])
print(f"Downsampled pooled train: {len(pooled_X_train)} samples")


In [ ]:
# ── Per-site preprocessing ──
client_train_pp = {}
client_test_pp = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
    print(f"  Site {site}: preprocessed")
print("Per-site preprocessing done.")

In [ ]:
# ── Load archived tuning from the with-D run (matched configuration) ──
# Instead of re-tuning the RF grid / thresholds on A/B/C only (which would add a
# second difference vs the with-D run), reuse the archived values verbatim.
import ast

_txt = Path(ARCHIVE_PARAMS).read_text()
_vals = {}
for _line in _txt.splitlines():
    if "=" in _line:
        _k, _, _v = _line.partition("=")
        _vals[_k.strip()] = _v.strip()

BEST_LR_C         = float(_vals["BEST_LR_C"])
BEST_MLP_LR       = float(_vals["BEST_MLP_LR"])
BEST_MLP_DH       = float(_vals["BEST_MLP_DH"])
BEST_LR_THRESH    = float(_vals["BEST_LR_THRESH"])
BEST_MLP_THRESH   = float(_vals["BEST_MLP_THRESH"])
RF_PARAMS         = ast.literal_eval(_vals["RF_PARAMS"])
BEST_RF_THRESH    = float(_vals["BEST_RF_THRESH"])
RF_TREES_PER_ROUND = RF_PARAMS["n_estimators"]

print(f"Archived tuning from {ARCHIVE_RUN}:")
print(f"  MLP: lr={BEST_MLP_LR:.2e} drop={BEST_MLP_DH} thresh={BEST_MLP_THRESH:.3f}")
print(f"  LR:  C={BEST_LR_C:.2e} thresh={BEST_LR_THRESH:.3f}")
print(f"  RF:  {RF_PARAMS} thresh={BEST_RF_THRESH:.3f}")


In [ ]:
# ── Dataset helper (RF grid + threshold tuning skipped: archived values reused) ──

class DatasetFromNumpy(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CENTRALIZED MLP (pooled data, pre-computed params)
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Centralized MLP ===")
# Pooled preprocessing
state_pool = fit_input_transform(pooled_X_train, "log1p+standardize")
X_pool_pp = apply_input_transform(pooled_X_train, state_pool)

# Preprocess all test sets with pooled state
pooled_test_sets = {}
for site in SITE_ORDER:
    X_te, y_te = client_test[site]
    pooled_test_sets[site] = (apply_input_transform(X_te, state_pool), y_te)
combined_X_test = np.concatenate([pooled_test_sets[s][0] for s in SITE_ORDER])
combined_y_test = np.concatenate([pooled_test_sets[s][1] for s in SITE_ORDER])

# Train centralized MLP
cent_model = SpectralAttentionMLP(
    input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256, 128),
    dropout_high=BEST_MLP_DH, dropout_low=BEST_MLP_DH/2.0, use_attention=False)
device = "cuda" if torch.cuda.is_available() else "cpu"
cent_model.to(device)
ds = DatasetFromNumpy(X_pool_pp, pooled_y_train)
dl = DataLoader(ds, batch_size=64, shuffle=True)
opt = torch.optim.AdamW(cent_model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
crit = nn.CrossEntropyLoss()
best_loss, best_sd, patience = float("inf"), None, 0
for ep in range(100):
    cent_model.train()
    for xb, yb in dl:
        if len(xb) <= 1: continue
        xb, yb = xb.to(device), yb.to(device)
        if ep < 10:
            for pg in opt.param_groups: pg["lr"] = BEST_MLP_LR * (ep + 1) / 10
        opt.zero_grad(); crit(cent_model(xb), yb).backward(); opt.step()
    if ep >= 10: sched.step()
    if ep % 5 == 0:
        cent_model.eval()
        with torch.no_grad():
            vl = sum(crit(cent_model(xb.to(device)), yb.to(device)).item() for xb, yb in dl) / len(dl)
        if vl < best_loss:
            best_loss = vl; best_sd = {k: v.cpu().clone() for k, v in cent_model.state_dict().items()}; patience = 0
        else:
            patience += 1
            if patience >= 3: break
if best_sd: cent_model.load_state_dict(best_sd)
cent_model.eval()

def mlp_predict_proba(model, X_np, dev=device):
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(dev)
    with torch.no_grad():
        return F.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

centralized_mlp_results = {}
for site in SITE_ORDER:
    X_tt, y_tt = pooled_test_sets[site]
    proba = mlp_predict_proba(cent_model, X_tt)
    centralized_mlp_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    centralized_mlp_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = mlp_predict_proba(cent_model, combined_X_test)
centralized_mlp_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= BEST_MLP_THRESH)
centralized_mlp_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
# A/B/C-only pooled (D is not trained in this arm)
combined_abc_X = np.concatenate([pooled_test_sets[s][0] for s in TRAIN_SITES])
combined_abc_y = np.concatenate([pooled_test_sets[s][1] for s in TRAIN_SITES])
proba_abc = mlp_predict_proba(cent_model, combined_abc_X)
centralized_mlp_results["AllABC_BalAcc"] = balanced_accuracy_score(combined_abc_y, proba_abc >= BEST_MLP_THRESH)
centralized_mlp_results["AllABC_AUC"] = roc_auc_score(combined_abc_y, proba_abc)
print("Centralized MLP results:")
for site in SITE_ORDER:
    print(f"  {site}: BalAcc={centralized_mlp_results[f'{site}_BalAcc']:.4f}  AUC={centralized_mlp_results[f'{site}_AUC']:.4f}")
print(f"  All: BalAcc={centralized_mlp_results['All_BalAcc']:.4f}  AUC={centralized_mlp_results['All_AUC']:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CENTRALIZED RF (pooled data, Option-A params)
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Centralized RF ===")
rf_cent = RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1)
rf_cent.fit(X_pool_pp, pooled_y_train)

centralized_rf_results = {}
for site in SITE_ORDER:
    X_tt, y_tt = pooled_test_sets[site]
    proba = rf_cent.predict_proba(X_tt)[:, 1]
    centralized_rf_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_RF_THRESH)
    centralized_rf_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = rf_cent.predict_proba(combined_X_test)[:, 1]
centralized_rf_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= BEST_RF_THRESH)
centralized_rf_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
combined_abc_X = np.concatenate([pooled_test_sets[s][0] for s in TRAIN_SITES])
combined_abc_y = np.concatenate([pooled_test_sets[s][1] for s in TRAIN_SITES])
proba_abc = rf_cent.predict_proba(combined_abc_X)[:, 1]
centralized_rf_results["AllABC_BalAcc"] = balanced_accuracy_score(combined_abc_y, proba_abc >= BEST_RF_THRESH)
centralized_rf_results["AllABC_AUC"] = roc_auc_score(combined_abc_y, proba_abc)
print("Centralized RF results:")
for site in SITE_ORDER:
    print(f"  {site}: BalAcc={centralized_rf_results[f'{site}_BalAcc']:.4f}  AUC={centralized_rf_results[f'{site}_AUC']:.4f}")
print(f"  All: BalAcc={centralized_rf_results['All_BalAcc']:.4f}  AUC={centralized_rf_results['All_AUC']:.4f}")

In [ ]:
# ── Model construction + serialisation helpers ──
def build_mlp():
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256, 128),
        dropout_high=BEST_MLP_DH, dropout_low=BEST_MLP_DH/2.0, use_attention=False)

def model_to_numpy(model):
    return [v.cpu().numpy() for v in model.state_dict().values()]

def numpy_to_model(model, params):
    sd = model.state_dict()
    for k, p in zip(sd.keys(), params):
        sd[k] = torch.tensor(p)
    model.load_state_dict(sd)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOCAL MLP TRAINING (mixing + FedProx proximal term)
# ═══════════════════════════════════════════════════════════════════════════

def train_local_mixed(model, X_np, y_np, n_mixes, proximal_mu, global_params):
    """Train model on (X_np, y_np) for n_mixes epochs, each with different seed.
    If n_mixes > 1, average state_dicts. Returns final model state_dict + avg loss."""
    collect_sds = []
    total_loss = 0.0
    for mix_i in range(n_mixes):
        seed = SEED + mix_i * 100 + int(proximal_mu * 1000)
        torch.manual_seed(seed); np.random.seed(seed)
        # Shuffle indices
        idx = np.random.permutation(len(X_np))
        X_shuf, y_shuf = X_np[idx], y_np[idx]
        ds = DatasetFromNumpy(X_shuf, y_shuf)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

        opt = torch.optim.AdamW(model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        model.train()
        batch_loss = 0.0; n_batch = 0
        for xb, yb in dl:
            if len(xb) <= 1: continue
            xb, yb = xb.to(FL_DEVICE), yb.to(FL_DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            if proximal_mu > 0 and global_params is not None:
                prox = sum((w - gw.to(FL_DEVICE)).norm(2) ** 2
                          for w, gw in zip(model.parameters(), global_params))
                loss = loss + (proximal_mu / 2.0) * prox
            loss.backward(); opt.step()
            batch_loss += loss.item(); n_batch += 1
        total_loss += batch_loss / n_batch
        collect_sds.append({k: v.cpu().clone() for k, v in model.state_dict().items()})

    # Average state dicts if multiple mixes
    if len(collect_sds) > 1:
        avg_sd = {}
        for k in collect_sds[0].keys():
            stacked = torch.stack([sd[k].float() for sd in collect_sds])
            avg_sd[k] = stacked.mean(0)
            # Preserve Long dtype for known integer keys
            if k.endswith("num_batches_tracked"):
                avg_sd[k] = avg_sd[k].to(torch.long) if avg_sd[k].dim() == 0 else avg_sd[k].round().to(torch.long)
        model.load_state_dict(avg_sd)
    return total_loss / n_mixes

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FED MLP CLIENT
# ═══════════════════════════════════════════════════════════════════════════

class FedMLPClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid
        self.X_train, self.y_train = X_train, y_train
        self.n_mixes = n_mixes(len(X_train))
        self.model = build_mlp().to(FL_DEVICE)

    def get_parameters(self, config):
        return model_to_numpy(self.model)

    def set_parameters(self, params):
        numpy_to_model(self.model, params)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        proximal_mu = float(config.get("proximal_mu", 0.0))
        global_copy = None
        if proximal_mu > 0:
            global_copy = [p.clone().detach() for p in self.model.parameters()]
        loss = train_local_mixed(self.model, self.X_train, self.y_train,
                                 self.n_mixes, proximal_mu, global_copy)
        return (self.get_parameters({}), len(self.X_train),
                {"train_loss": loss, "n_mixes": self.n_mixes})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FED LR CLIENT
# ═══════════════════════════════════════════════════════════════════════════

class FedLRClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        nf = X_train.shape[1]
        self.model = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga",
                                        max_iter=1, warm_start=True,
                                        class_weight="balanced", random_state=SEED)
        self.model.classes_ = np.array([0, 1])
        self.model.coef_ = np.zeros((1, nf)); self.model.intercept_ = np.zeros(1)

    def get_parameters(self, config):
        return [self.model.coef_.ravel(), self.model.intercept_]

    def set_parameters(self, params):
        self.model.coef_ = params[0].reshape(1, -1); self.model.intercept_ = params[1]

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.model.fit(self.X_train, self.y_train)
        tl = log_loss(self.y_train, self.model.predict_proba(self.X_train)[:, 1], labels=[0, 1])
        return (self.get_parameters({}), len(self.X_train), {"train_loss": float(tl), "num_examples": len(self.X_train)})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FED RF CLIENT + TREE COLLECTION STRATEGY
# ═══════════════════════════════════════════════════════════════════════════

def trees_to_array(estimators):
    buf = io.BytesIO(); joblib.dump(estimators, buf); buf.seek(0)
    return np.frombuffer(buf.read(), dtype=np.uint8)

def array_to_trees(arr):
    return joblib.load(io.BytesIO(arr.tobytes()))

class TreeCollectionFedAvg(fl.server.strategy.FedAvg):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._accumulated_trees = []

    def aggregate_fit(self, server_round, results, failures):
        if not results: return None, {}
        for _, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            if len(ndarrays[0]) > 0:
                self._accumulated_trees.extend(array_to_trees(ndarrays[0]))
        combined = trees_to_array(self._accumulated_trees)
        aggregated = fl.common.ndarrays_to_parameters([combined])
        metrics = {"total_trees": len(self._accumulated_trees)}
        if self.fit_metrics_aggregation_fn:
            metrics.update(self.fit_metrics_aggregation_fn(
                [fit_res.metrics for _, fit_res in results]))
        return aggregated, metrics

class FedRFClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train

    def get_parameters(self, config):
        return [np.array([], dtype=np.uint8)]

    def fit(self, parameters, config):
        server_round = int(config.get("current_round", 0))
        rf = RandomForestClassifier(**RF_PARAMS,
            random_state=SEED + int(self.cid) + 100 * server_round,
            n_jobs=-1, warm_start=True)
        rf.fit(self.X_train, self.y_train)
        new_trees = list(rf.estimators_)
        return ([trees_to_array(new_trees)], len(self.X_train),
                {"n_new_trees": len(new_trees), "num_examples": len(self.X_train)})

In [ ]:
# ── Strategy classes (checkpoint saving disabled in the ablation arm) ──

class CheckpointFedAvg(fl.server.strategy.FedAvg):
    """FedAvg for the no-D arm: plain aggregation, no checkpoint files."""
    def __init__(self, strategy_name, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.strategy_name = strategy_name
        self.model_dir = Path(model_dir) / strategy_name


In [ ]:
# ── LR aggregation (checkpoint saving disabled in the ablation arm) ──

class CheckpointLRFedAvg(fl.server.strategy.FedAvg):
    """FedAvg for FedLR in the no-D arm: plain aggregation, no checkpoint files."""
    def __init__(self, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.model_dir = Path(model_dir) / "fedavg_lr"


In [ ]:
# ── RF tree collection (checkpoint saving disabled in the ablation arm) ──

class CheckpointTreeCollection(TreeCollectionFedAvg):
    """Tree collection for the no-D arm: grows the ensemble, no checkpoint files."""
    def __init__(self, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.model_dir = Path(model_dir) / "fedrf"


In [ ]:
# ── Server-side evaluation functions ──

def get_mlp_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        m = build_mlp(); numpy_to_model(m, parameters); m.to(FL_DEVICE); m.eval()
        record = {"round": server_round}
        ap, al = [], []
        site_probas, site_targets = {}, {}
        ce_losses = []; ce_weights = []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            X_t = torch.tensor(X_tt, dtype=torch.float32).to(FL_DEVICE)
            y_t = torch.tensor(y_tt, dtype=torch.long).to(FL_DEVICE)
            with torch.no_grad():
                logits = m(X_t)
                loss = F.cross_entropy(logits, y_t).item()
            ce_losses.append(loss * len(y_tt)); ce_weights.append(len(y_tt))
            proba = F.softmax(logits, dim=1).cpu().numpy()[:, 1]
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
            site_probas[site] = proba; site_targets[site] = y_tt
        record["val_loss"] = float(sum(ce_losses) / sum(ce_weights)) if sum(ce_weights) > 0 else 0.0
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        abc_p = np.concatenate([site_probas[s] for s in TRAIN_SITES])
        abc_y = np.concatenate([site_targets[s] for s in TRAIN_SITES])
        record["AllABC_BalAcc"] = float(balanced_accuracy_score(abc_y, abc_p >= threshold))
        record["AllABC_AUC"] = float(roc_auc_score(abc_y, abc_p))
        # Save metrics.json
        rd = Path(MODEL_DIR) / '__current_strategy__' / f"round_{server_round:03d}"
        if rd.exists():
            import json as _json
            with open(rd / "metrics.json", "w") as f:
                _json.dump(record, f)
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

def get_lr_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        lr = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="lbfgs",
                                class_weight="balanced", max_iter=5000)
        lr.classes_ = np.array([0, 1])
        lr.coef_ = parameters[0].reshape(1, -1); lr.intercept_ = parameters[1]
        record = {"round": server_round}
        ap, al = [], []
        site_probas, site_targets = {}, {}
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = lr.predict_proba(X_tt)[:, 1]
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
            site_probas[site] = proba; site_targets[site] = y_tt
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        abc_p = np.concatenate([site_probas[s] for s in TRAIN_SITES])
        abc_y = np.concatenate([site_targets[s] for s in TRAIN_SITES])
        record["AllABC_BalAcc"] = float(balanced_accuracy_score(abc_y, abc_p >= threshold))
        record["AllABC_AUC"] = float(roc_auc_score(abc_y, abc_p))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

def get_rf_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        if len(parameters[0]) == 0:
            return (1.0, {"All_BalAcc": 0.5})
        trees = array_to_trees(parameters[0])
        rf = RandomForestClassifier(**RF_PARAMS, n_jobs=-1)
        rf.estimators_ = trees; rf.n_classes_ = 2
        rf.classes_ = np.array([0, 1]); rf.n_outputs_ = 1
        record = {"round": server_round, "n_trees": len(trees)}
        ap, al = [], []
        site_probas, site_targets = {}, {}
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = rf.predict_proba(X_tt)[:, 1]
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
            site_probas[site] = proba; site_targets[site] = y_tt
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        abc_p = np.concatenate([site_probas[s] for s in TRAIN_SITES])
        abc_y = np.concatenate([site_targets[s] for s in TRAIN_SITES])
        record["AllABC_BalAcc"] = float(balanced_accuracy_score(abc_y, abc_p >= threshold))
        record["AllABC_AUC"] = float(roc_auc_score(abc_y, abc_p))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

In [ ]:
# ── Client factories ──
def mlp_client_fn(cid):
    site = TRAIN_SITES[int(cid)]
    X_tr, y_tr = client_train_pp[site]
    return FedMLPClient(cid, X_tr, y_tr).to_client()

def lr_client_fn(cid):
    site = TRAIN_SITES[int(cid)]
    X_tr, y_tr = client_train_pp[site]
    return FedLRClient(cid, X_tr, y_tr).to_client()

def rf_client_fn(cid):
    site = TRAIN_SITES[int(cid)]
    X_tr, y_tr = client_train_pp[site]
    return FedRFClient(cid, X_tr, y_tr).to_client()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN: FedAvg MLP
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== FedAvg MLP ===")
eval_hist_fedavg = []
_eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_fedavg)

fedavg_train_hist = []
round_ctr = {"r": 0}
def _fit_agg(metrics):
    agg = weighted_avg_loss(metrics)
    round_ctr["r"] += 1
    fedavg_train_hist.append({"round": round_ctr["r"], "All_train_loss": float(agg.get("avg_train_loss", np.nan))})
    return agg

strategy = CheckpointFedAvg("fedavg_mlp", MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0,
    min_fit_clients=len(TRAIN_SITES), min_evaluate_clients=len(TRAIN_SITES), min_available_clients=len(TRAIN_SITES), fit_metrics_aggregation_fn=_fit_agg,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp())))

strategy.model_dir = Path(MODEL_DIR) / "fedavg_mlp"  # re-set after init

fl.simulation.start_simulation(
    client_fn=mlp_client_fn, num_clients=len(TRAIN_SITES),
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedavg_hist = eval_hist_fedavg.copy()
print(f"FedAvg MLP done. {len(fedavg_hist)} rounds.")

In [ ]:
# ── FedProx skipped in the Site-D ablation arm (unstable at tuned μ) ──
fedprox_histories = {}
fedprox_train_histories = {}
print("FedProx skipped in the no-D ablation arm.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN: FedAvg LR
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== FedAvg LR ===")
eval_hist_fedlr = []
_eval_fn = get_lr_eval_fn(client_test_pp, BEST_LR_THRESH, eval_hist_fedlr)

fedlr_train_hist = []
lr_round_ctr = {"r": 0}
def lr_fit_agg(metrics):
    agg = weighted_avg_loss(metrics)
    lr_round_ctr["r"] += 1
    fedlr_train_hist.append({"round": lr_round_ctr["r"], "All_train_loss": float(agg.get("avg_train_loss", np.nan))})
    return agg

nr_feat = client_train_pp[SITE_ORDER[0]][0].shape[1]
lr_init = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga",
                              max_iter=1, warm_start=True, class_weight="balanced", random_state=SEED)
lr_init.classes_ = np.array([0, 1]); lr_init.coef_ = np.zeros((1, nr_feat)); lr_init.intercept_ = np.zeros(1)
lr_init_params = [lr_init.coef_.ravel(), lr_init.intercept_]

strategy = CheckpointLRFedAvg(MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0,
    min_fit_clients=len(TRAIN_SITES), min_evaluate_clients=len(TRAIN_SITES), min_available_clients=len(TRAIN_SITES), fit_metrics_aggregation_fn=lr_fit_agg,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters(lr_init_params))

strategy.model_dir = Path(MODEL_DIR) / "fedavg_lr"
fl.simulation.start_simulation(
    client_fn=lr_client_fn, num_clients=len(TRAIN_SITES),
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedlr_hist = eval_hist_fedlr.copy()
print(f"FedAvg LR done. {len(fedlr_hist)} rounds.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN: FedRF (tree collection)
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== FedRF (Tree Collection) ===")
eval_hist_fedrf = []
_eval_fn = get_rf_eval_fn(client_test_pp, BEST_RF_THRESH, eval_hist_fedrf)

strategy = CheckpointTreeCollection(MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0,
    min_fit_clients=len(TRAIN_SITES), min_evaluate_clients=len(TRAIN_SITES), min_available_clients=len(TRAIN_SITES), fit_metrics_aggregation_fn=weighted_avg_loss,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters([np.array([], dtype=np.uint8)]))

strategy.model_dir = Path(MODEL_DIR) / "fedrf"
fl.simulation.start_simulation(
    client_fn=rf_client_fn, num_clients=len(TRAIN_SITES),
    config=fl.server.ServerConfig(num_rounds=NUM_RF_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedrf_hist = eval_hist_fedrf.copy()
print(f"FedRF done. {len(fedrf_hist)} rounds.")

In [ ]:
# ── Assemble final results DataFrame ──
def best_metrics(h):
    """Return (metrics_dict, peak_round) for round with highest All_BalAcc."""
    if not h: return {}, 0
    skip0 = h[1:]  # skip round 0 (initialisation)
    best = max(skip0, key=lambda r: r.get("All_BalAcc", 0))
    return best, int(best.get("round", 0))

rows = []

def add_row(method, cs_source=None, fed_source=None, fed_hist=None):
    r = {"Method": method}
    peak_round = ""
    for site in SITE_ORDER:
        if cs_source:
            r[f"{site}_BalAcc"] = cs_source.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = cs_source.get(f"{site}_AUC", np.nan)
        elif fed_hist is not None:
            lm, pr = best_metrics(fed_hist)
            r[f"{site}_BalAcc"] = lm.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = lm.get(f"{site}_AUC", np.nan)
            peak_round = f" (r{pr})"
    if fed_hist is not None:
        lm, pr = best_metrics(fed_hist)
        r["All_BalAcc"] = lm.get("All_BalAcc", np.nan)
        r["All_AUC"] = lm.get("All_AUC", np.nan)
        r["AllABC_BalAcc"] = lm.get("AllABC_BalAcc", np.nan)
        r["AllABC_AUC"] = lm.get("AllABC_AUC", np.nan)
        r["Peak_Round"] = int(pr)
    elif cs_source:
        r["All_BalAcc"] = cs_source.get("All_BalAcc", np.nan)
        r["All_AUC"] = cs_source.get("All_AUC", np.nan)
        r["AllABC_BalAcc"] = cs_source.get("AllABC_BalAcc", np.nan)
        r["AllABC_AUC"] = cs_source.get("AllABC_AUC", np.nan)
        r["Peak_Round"] = 0
    r["Label"] = method + peak_round  # for display
    rows.append(r)

add_row("Centralized MLP", cs_source=centralized_mlp_results)
add_row("Centralized RF", cs_source=centralized_rf_results)
add_row("FL FedAvg (MLP)", fed_hist=fedavg_hist)
for mu in sorted(fedprox_histories.keys()):
    add_row(f"FL FedProx mu={mu} (MLP)", fed_hist=fedprox_histories[mu])
add_row("FL FedAvg (LR)", fed_hist=fedlr_hist)
add_row("FL FedRF (Trees)", fed_hist=fedrf_hist)
# Cross-site baselines are not run in the no-D arm (D untrained; A→B/C covered elsewhere)

df_results = pd.DataFrame(rows)
cols = ["Method", "Label", "Peak_Round"] + [f"{s}_BalAcc" for s in SITE_ORDER] + ["All_BalAcc"]
print(df_results[cols].to_string(index=False))

In [ ]:
# ── Convergence plot ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Panel 1: FedAvg per-site
ax = axes[0]
for site in SITE_ORDER:
    vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedavg_hist]
    ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
vals_all = [h.get("All_BalAcc", np.nan) for h in fedavg_hist]
ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
ax.set_title("FedAvg MLP — Per-Site Convergence"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

# Panel 2: All FL methods, All BalAcc
ax = axes[1]
plots_data = [
    ("FedAvg MLP", fedavg_hist, "#ff7f0e", "-"),
]
# Add all FedProx variants dynamically
fedprox_colors = {0.3: "#d62728", 0.5: "#9467bd", 0.7: "#8c564b", 0.01: "#e377c2", 0.1: "#7f7f7f"}
for mu in sorted(fedprox_histories.keys()):
    plots_data.append((f"FedProx mu={mu}", fedprox_histories[mu],
                       fedprox_colors.get(mu, "#d62728"), "--"))
plots_data += [
    ("FedAvg LR", fedlr_hist, "#1f77b4", "-."),
    ("FedRF", fedrf_hist, "#2ca02c", "-"),
]
for label, hist, c, ls in plots_data:
    vals = [h.get("All_BalAcc", np.nan) for h in hist]
    ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=2, label=label)
    # Annotate peak round
    skip0 = hist[1:]
    if skip0:
        best = max(skip0, key=lambda h: h.get("All_BalAcc", 0))
        pr = int(best.get("round", 0))
        ax.axvline(pr, color=c, ls=':', alpha=0.4, lw=1)
        ax.annotate(f"r{pr}", (pr, best.get("All_BalAcc", 0)),
                    textcoords="offset points", xytext=(3, 5), fontsize=7, color=c)
ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=2,
           label=f'Centralized MLP')
ax.axhline(centralized_rf_results["All_BalAcc"], color='gray', ls='--', lw=2,
           label=f'Centralized RF')
ax.set_title("All Methods — All-Site BalAcc (peak rounds marked)"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

ax = axes[2]
plots = [("FedAvg MLP", fedavg_hist, "#ff7f0e", "-")]
for mu in FEDPROX_MUS:
    if mu in fedprox_histories: plots.append((f"FedProx mu={mu}", fedprox_histories[mu], "#d62728", "--"))
plots.append(("FedAvg LR", fedlr_hist, "#1f77b4", "-."))
if fedrf_hist: plots.append(("FedRF", fedrf_hist, "#2ca02c", "-"))
for label, hist, c, ls in plots:
    vals = [h.get("val_loss", np.nan) for h in hist if "val_loss" in h]
    if vals: ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=1.5, alpha=0.55, label=f"{label} (val)")
# Training loss (empty for runs without train-loss logging)
train_plots = [("FedAvg MLP", fedavg_train_hist, "#ff7f0e", "-")]
for mu in FEDPROX_MUS:
    if mu in fedprox_train_histories: train_plots.append((f"FedProx mu={mu}", fedprox_train_histories[mu], "#d62728", "--"))
train_plots.append(("FedAvg LR", fedlr_train_hist, "#1f77b4", "-."))
for label, th, c, ls in train_plots:
    vals = [r.get("All_train_loss", np.nan) for r in th if "All_train_loss" in r]
    if vals: ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=1.8, label=f"{label} (train)")
ax.set_title("Train / Val Loss"); ax.set_xlabel("Round"); ax.set_ylabel("CE Loss")
ax.legend(fontsize=6); ax.grid(True, ls='--', alpha=0.5)

fig.suptitle(f"{DRUG_NAME} — Federated Learning Convergence", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "convergence.pdf", bbox_inches="tight")
plt.savefig(OUT_DIR / "loss_curves.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── FedProx Per-Site Convergence ──
for mu in FEDPROX_MUS:
    if mu not in fedprox_histories:
        continue
    fig, ax = plt.subplots(figsize=(8, 5))
    for site in SITE_ORDER:
        vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedprox_histories[mu]]
        ax.plot(range(1, len(vals)+1), vals, marker=".", label=f"Site {site}")
    vals_all = [h.get("All_BalAcc", np.nan) for h in fedprox_histories[mu]]
    ax.plot(range(1, len(vals_all)+1), vals_all, "k-", lw=2, label="All")
    ax.axhline(centralized_mlp_results["All_BalAcc"], color="gray", ls=":", lw=1.5)
    ax.set_title(f"{DRUG_NAME} — FedProx μ={mu} Per-Site Convergence")
    ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
    ax.legend(fontsize=8); ax.grid(True, ls="--", alpha=0.5); ax.set_ylim(0.3, 1.0)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "convergence_fedprox.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# ── FedLR Per-Site Convergence ──
fig, ax = plt.subplots(figsize=(8, 5))
for site in SITE_ORDER:
    vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedlr_hist]
    ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
vals_all = [h.get("All_BalAcc", np.nan) for h in fedlr_hist]
ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=1.5)
ax.set_title(f"{DRUG_NAME} — FedLR Per-Site Convergence")
ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)
plt.tight_layout()
plt.savefig(OUT_DIR / "convergence_fedlr.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── FedRF Per-Site Convergence ──
fig, ax = plt.subplots(figsize=(8, 5))
for site in SITE_ORDER:
    vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedrf_hist]
    ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
vals_all = [h.get("All_BalAcc", np.nan) for h in fedrf_hist]
ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=1.5)
ax.set_title(f"{DRUG_NAME} — FedRF Per-Site Convergence")
ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)
plt.tight_layout()
plt.savefig(OUT_DIR / "convergence_fedrf.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── FedProx retry cell intentionally skipped in the no-D ablation arm ──
print("FedProx retry skipped.")


In [ ]:
# ── Heatmap: Balanced Accuracy ──
ba_data = {}
for _, r in df_results.iterrows():
    ba_data[r["Label"]] = {f"Site {s}": r[f"{s}_BalAcc"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_BalAcc", np.nan)):
        ba_data[r["Label"]]["All"] = r["All_BalAcc"]
    ba_data[r["Label"]]["AllABC"] = r.get("AllABC_BalAcc", np.nan)

df_ba_hm = pd.DataFrame(ba_data).T
df_ba_hm = df_ba_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All", "AllABC"] if c in df_ba_hm.columns]]

fig, ax = plt.subplots(figsize=(10, max(4, len(df_ba_hm)*0.5)))
sns.heatmap(df_ba_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title(f"{DRUG_NAME} — Balanced Accuracy", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_balacc.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Heatmap: AUC ──
auc_data = {}
for _, r in df_results.iterrows():
    auc_data[r["Label"]] = {f"Site {s}": r[f"{s}_AUC"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_AUC", np.nan)):
        auc_data[r["Label"]]["All"] = r["All_AUC"]
    auc_data[r["Label"]]["AllABC"] = r.get("AllABC_AUC", np.nan)

df_auc_hm = pd.DataFrame(auc_data).T
df_auc_hm = df_auc_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All", "AllABC"] if c in df_auc_hm.columns]]

fig, ax = plt.subplots(figsize=(10, max(4, len(df_auc_hm)*0.5)))
sns.heatmap(df_auc_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC"}, ax=ax)
ax.set_title(f"{DRUG_NAME} — AUC-ROC", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_auc.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Save results ──
df_results.to_csv(OUT_DIR / "final_results.csv", index=False)
pd.DataFrame(fedavg_hist).to_csv(OUT_DIR / "fedavg_per_round.csv", index=False)
if fedavg_train_hist:
    pd.DataFrame(fedavg_train_hist).to_csv(OUT_DIR / "fedavg_train_loss_per_round.csv", index=False)
pd.DataFrame(fedlr_hist).to_csv(OUT_DIR / "fedlr_per_round.csv", index=False)
if fedlr_train_hist:
    pd.DataFrame(fedlr_train_hist).to_csv(OUT_DIR / "fedlr_train_loss_per_round.csv", index=False)
pd.DataFrame(fedrf_hist).to_csv(OUT_DIR / "fedrf_per_round.csv", index=False)
for mu in sorted(fedprox_histories.keys()):
    pd.DataFrame(fedprox_histories[mu]).to_csv(OUT_DIR / f"fedprox_mu{mu}_per_round.csv", index=False)
    if mu in fedprox_train_histories:
        pd.DataFrame(fedprox_train_histories[mu]).to_csv(OUT_DIR / f"fedprox_mu{mu}_train_loss_per_round.csv", index=False)

# Save best params used
with open(OUT_DIR / "best_params_used.txt", "w") as f:
    f.write(f"DRUG={DRUG_NAME}\n")
    f.write(f"BEST_LR_C={BEST_LR_C}\n")
    f.write(f"BEST_MLP_LR={BEST_MLP_LR}\n")
    f.write(f"BEST_MLP_DH={BEST_MLP_DH}\n")
    f.write(f"BEST_LR_THRESH={BEST_LR_THRESH}\n")
    f.write(f"BEST_MLP_THRESH={BEST_MLP_THRESH}\n")
    f.write(f"RF_PARAMS={RF_PARAMS}\n")
    f.write(f"BEST_RF_THRESH={BEST_RF_THRESH}\n")
    f.write(f"ABLATION=noD\n")
    f.write(f"TRAIN_SITES={TRAIN_SITES}\n")
    f.write(f"SOURCE_RUN={ARCHIVE_RUN}\n")

print("\n" + "="*60)
print(f"  Done. Results in {OUT_DIR.resolve()}")
print(f"  Models in {MODEL_DIR.resolve()}")
for f in sorted(OUT_DIR.glob("*")):
    print(f"    {f.name}")
print("\nModel directories:")
for d in sorted(MODEL_DIR.glob("*")):
    ndirs = len(list(d.glob("round_*")))
    print(f"    {d.name}/ ({ndirs} rounds)")

In [ ]:
# ── Local-only run: results stay in ./results (no Drive archival) ──
print(f"No-D ablation run complete. Results in {{OUT_DIR.resolve()}}")


---
**Done.** Site-D ablation (no-D arm) for **Ceftazidime** complete.

| Output | Location |
|---|---|
| Results | `Site-D-Ablation/Ceftazidime/results/` |
| Mirror (with-D) | `Drugs/Ceftazidime/03-Run/results/` |
| Notebook | `federated_noD.ipynb` |
